## 1. Data Ingestion

The dataset contains UK Police street crime data for Dorset, Wiltshire, Gloucestershire and Warwickshire, covering January 2024 to December 2025. 

`pathlib` is used to find the CSV files automatically rather than importing them individually. 

The pipeline checks that every expected force and month is present before processing. The date range is determined from the filenames, allowing additional months to be included without changing the code.

In [1]:
# Block checks that all expected source files are present before reading the data

from pathlib import Path
import pandas as pd

data_folder = Path("crime_data")
crime_files = sorted(data_folder.glob("*.csv"))
assert crime_files, "No crime CSV files found"

#Expected police forces (additional forces can be added) 
forces = [
    "dorset",
    "gloucestershire",
    "warwickshire",
    "wiltshire"]

# Extract dates and forces from filenames
file_info = pd.DataFrame([{
    "month": file.name[:7],
    "force": file.stem[8:].removesuffix("-street"),
    "filename": file.name}

    for file in crime_files])

# Determine the date range from the available files
months = pd.period_range(start=file_info["month"].min(), end=file_info["month"].max(), freq="M")

# Generate expected force/month combinations
expected = pd.MultiIndex.from_product([months.astype(str), forces], names=["month", "force"])
actual = pd.MultiIndex.from_frame(file_info[["month", "force"]])

# Check for missing unexpected or duplicate files 
missing = expected.difference(actual)
unexpected = actual.difference(expected)

assert len(missing) == 0, f"Missing files: {missing.tolist()}"
assert len(unexpected) == 0, f"Unexpected files: {unexpected.tolist()}"
assert not actual.has_duplicates, "Duplicate force/month files detected"

# Print results 
print(f"Found {len(crime_files)} crime files")
print(f"Police forces: {len(forces)}")
print(f"Months: {len(months)}")
print(f"Period: {months.min()} to {months.max()}")
print("All expected force/month combinations present.")

Found 96 crime files
Police forces: 4
Months: 24
Period: 2024-01 to 2025-12
All expected force/month combinations present.


### Initial Source Data Inspection

A single file (January 2024 Dorset) is loaded to inspect the columns, data types, missing values and crime categories before processing the full dataset.

This helps identify potential data quality issues and confirm the expected structure.

In [2]:
sample = pd.read_csv(crime_files[0])

print(sample.shape)
print(sample.columns.tolist())
sample.head()

(4802, 12)
['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2024-01,Dorset Police,Dorset Police,-1.946730,50.779999,On or near Moortown Drive,E01015431,"Bournemouth, Christchurch and Poole 001A",Anti-social behaviour,NaN,NaN
1,NaN,2024-01,Dorset Police,Dorset Police,-1.946730,50.779999,On or near Moortown Drive,E01015431,"Bournemouth, Christchurch and Poole 001A",Anti-social behaviour,NaN,NaN
2,NaN,2024-01,Dorset Police,Dorset Police,-1.946730,50.779999,On or near Moortown Drive,E01015431,"Bournemouth, Christchurch and Poole 001A",Anti-social behaviour,NaN,NaN
3,603af064b2253be8fb4e28ece53d01802bcdc033d584ea...,2024-01,Dorset Police,Dorset Police,-1.959419,50.786389,On or near Mountjoy Close,E01015431,"Bournemouth, Christchurch and Poole 001A",Vehicle crime,Investigation complete; no suspect identified,NaN
4,9caf624adcd662a3e0c500d01cd6bcfe422b4c3e822fde...,2024-01,Dorset Police,Dorset Police,-1.964046,50.784025,On or near Cockerell Close,E01015432,"Bournemouth, Christchurch and Poole 001B",Violence and sexual offences,Unable to prosecute suspect,NaN


In [3]:
sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4802 entries, 0 to 4801
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               3844 non-null   object 
 1   Month                  4802 non-null   object 
 2   Reported by            4802 non-null   object 
 3   Falls within           4802 non-null   object 
 4   Longitude              4788 non-null   float64
 5   Latitude               4788 non-null   float64
 6   Location               4802 non-null   object 
 7   LSOA code              4788 non-null   object 
 8   LSOA name              4788 non-null   object 
 9   Crime type             4802 non-null   object 
 10  Last outcome category  3844 non-null   object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 450.3+ KB


In [4]:
sample.isna().sum()

Crime ID                  958
Month                       0
Reported by                 0
Falls within                0
Longitude                  14
Latitude                   14
Location                    0
LSOA code                  14
LSOA name                  14
Crime type                  0
Last outcome category     958
Context                  4802
dtype: int64

In [5]:
sample["Crime type"].value_counts()

Crime type
Violence and sexual offences    1656
Anti-social behaviour            958
Criminal damage and arson        417
Vehicle crime                    354
Shoplifting                      350
Other theft                      336
Public order                     262
Burglary                         211
Drugs                             80
Bicycle theft                     52
Other crime                       43
Possession of weapons             42
Robbery                           21
Theft from the person             20
Name: count, dtype: int64

In [6]:
sample.groupby("Crime type")["Crime ID"].apply(lambda x: x.isna().sum())

Crime type
Anti-social behaviour           958
Bicycle theft                     0
Burglary                          0
Criminal damage and arson         0
Drugs                             0
Other crime                       0
Other theft                       0
Possession of weapons             0
Public order                      0
Robbery                           0
Shoplifting                       0
Theft from the person             0
Vehicle crime                     0
Violence and sexual offences      0
Name: Crime ID, dtype: int64

In [7]:
# Check all source files have the same columns
expected_columns = sample.columns.tolist()

for file in crime_files:
    columns = pd.read_csv(file, nrows=0).columns.tolist()
    assert columns == expected_columns, f"Column mismatch: {file.name}"

print("All source file schemas match.")

All source file schemas match.


### Batch Ingestion Using SQLite

Once the source files have been checked, Pandas reads them in chunks of 10,000 rows and stores them in a temporary SQLite database. This avoids keeping a separate DataFrame for every CSV file in memory.

The number of records stored in SQLite is checked against the number loaded from the files. The combined data is then loaded into Pandas for checks that require comparing records across files.

The SQLite connection is closed and the temporary database is deleted after use.

In [8]:
import sqlite3
from tempfile import TemporaryDirectory

total_loaded = 0

with TemporaryDirectory() as temp_dir:
    db_path = Path(temp_dir) / "crime_staging.db"

    conn = sqlite3.connect(db_path)

    try:
        for file in crime_files:
            file_rows = 0

            for batch in pd.read_csv(file, chunksize=10000):
                batch.to_sql("crime_records",conn,if_exists="append",index=False)
                file_rows += len(batch)

            total_loaded += file_rows

        # Check the SQLite row count matches the number of records loaded
        staged_rows = conn.execute("SELECT COUNT(*) FROM crime_records").fetchone()[0]

        print("Staged records:", staged_rows)
        print("Loaded records:", total_loaded)
        assert staged_rows == total_loaded

        # Load the combined records for checks across files
        crime_df = pd.read_sql_query("SELECT * FROM crime_records ORDER BY rowid",conn)

    finally:
        conn.close()

print(f"Files processed: {len(crime_files)}")
print(f"Records loaded: {total_loaded:,}")
print(f"Columns: {len(crime_df.columns)}")

assert len(crime_df) == total_loaded

Staged records: 437271
Loaded records: 437271
Files processed: 96
Records loaded: 437,271
Columns: 12


### Full Dataset Inspection

The initial checks are repeated on the combined dataset to determine whether the issues identified in the sample are present across all four police forces.

In [9]:
print(crime_df["Reported by"].value_counts())
print(crime_df["Month"].nunique())

Reported by
Gloucestershire Constabulary    117553
Dorset Police                   117392
Wiltshire Police                102071
Warwickshire Police             100255
Name: count, dtype: int64
24


In [10]:
crime_df.groupby("Crime type")["Crime ID"].apply(lambda x: x.isna().sum())

Crime type
Anti-social behaviour           91587
Bicycle theft                       0
Burglary                            0
Criminal damage and arson           0
Drugs                               0
Other crime                         0
Other theft                         0
Possession of weapons               0
Public order                        0
Robbery                             0
Shoplifting                         0
Theft from the person               0
Vehicle crime                       0
Violence and sexual offences        0
Name: Crime ID, dtype: int64

## 2. Data Quality Assessment

Initial inspection of the January 2024 Dorset file identified 958 missing Crime IDs, all belonging to antisocial behaviour (ASB).

Across the full dataset of 437,271 records 91,587 Crime IDs were missing. All were ASB records, suggesting that the missing identifiers reflect how these incidents are recorded rather than a data quality error.

These records will be retained with ASB distinguished from recorded crime.

In [11]:
# Count identical rows and repeated Crime IDs
print("Duplicate rows:", crime_df.duplicated().sum())
print("Duplicate Crime IDs:", crime_df.loc[crime_df["Crime ID"].notna(), "Crime ID"].duplicated().sum())

Duplicate rows: 25161
Duplicate Crime IDs: 580


In [12]:
# Check which crime types contain identical rows 
duplicates = crime_df[crime_df.duplicated()]
duplicates["Crime type"].value_counts()

Crime type
Anti-social behaviour           25145
Violence and sexual offences        5
Public order                        4
Other theft                         2
Burglary                            1
Shoplifting                         1
Drugs                               1
Possession of weapons               1
Other crime                         1
Name: count, dtype: int64

In [13]:
# Check for exact duplicate rows and repeated Crime IDs
crime_df[crime_df["Crime ID"].notna()].duplicated(subset=["Crime ID", "Month"]).sum()

574

In [14]:
# Check which crime types contain duplicate rows
repeated_crimes = crime_df[crime_df["Crime ID"].notna()& crime_df.duplicated(subset=["Crime ID", "Month"],keep=False)]

repeated_crimes.sort_values(["Crime ID", "Month"]).head(10)

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
202227,003cc8fa9fbf60638561f5bba41abef7c241bfff8d88da...,2024-11,Wiltshire Police,Wiltshire Police,-2.208689,51.302944,On or near Shopping Area,E01032092,Wiltshire 035A,Shoplifting,Status update unavailable,None
202229,003cc8fa9fbf60638561f5bba41abef7c241bfff8d88da...,2024-11,Wiltshire Police,Wiltshire Police,-2.207533,51.304115,On or near Alderton Way,E01032092,Wiltshire 035A,Shoplifting,Court result unavailable,None
231821,005e9594397b9137e709e5a33bd1f4e787719335e43f85...,2025-01,Wiltshire Police,Wiltshire Police,-1.709896,51.625137,On or near Stranks Close,E01015520,Swindon 001B,Violence and sexual offences,Offender given a caution,None
231991,005e9594397b9137e709e5a33bd1f4e787719335e43f85...,2025-01,Wiltshire Police,Wiltshire Police,-1.807467,51.586315,On or near Boscombe Road,E01015528,Swindon 006B,Violence and sexual offences,Status update unavailable,None
199338,007dad70f462f0cbb2325c06f72f3dffb5a37efd23b67a...,2024-11,Wiltshire Police,Wiltshire Police,-1.706898,51.639767,On or near Sevenfields,E01015522,Swindon 001D,Violence and sexual offences,Unable to prosecute suspect,None
199733,007dad70f462f0cbb2325c06f72f3dffb5a37efd23b67a...,2024-11,Wiltshire Police,Wiltshire Police,-1.777728,51.569799,On or near Handel Street,E01015510,Swindon 010C,Violence and sexual offences,Status update unavailable,None
435206,00866615f55127858e68c0fa21868f04f9098c292870d5...,2025-12,Wiltshire Police,Wiltshire Police,-1.548583,51.452675,On or near Park/Open Space,E01031873,Wiltshire 012C,Other theft,Status update unavailable,None
435207,00866615f55127858e68c0fa21868f04f9098c292870d5...,2025-12,Wiltshire Police,Wiltshire Police,-1.548583,51.452675,On or near Park/Open Space,E01031873,Wiltshire 012C,Other theft,Investigation complete; no suspect identified,None
328221,020607568320eadc5848a0b42f0daf8b705341a3174121...,2025-07,Dorset Police,Dorset Police,-1.966736,50.750825,On or near Parking Area,E01015415,"Bournemouth, Christchurch and Poole 013D",Violence and sexual offences,Status update unavailable,None
328386,020607568320eadc5848a0b42f0daf8b705341a3174121...,2025-07,Dorset Police,Dorset Police,-1.965392,50.747569,On or near Ashdown Close,E01015407,"Bournemouth, Christchurch and Poole 016A",Violence and sexual offences,Investigation complete; no suspect identified,None


In [15]:
crime_df[crime_df["Crime ID"].notna()].groupby(["Crime ID", "Month"])["Crime type"].nunique().value_counts()

Crime type
1    345105
2         5
Name: count, dtype: int64

In [16]:
category_counts = crime_df[crime_df["Crime ID"].notna()].groupby(["Crime ID", "Month"])["Crime type"].nunique()

conflicting_ids = category_counts[category_counts > 1].index

crime_df.set_index(["Crime ID", "Month"]).loc[conflicting_ids, ["Reported by", "Crime type", "Last outcome category"]]

Reported by  \
Crime ID                                           Month                       
2083bb0bc793766a6c5df56889ee37e2ff5a3c7c18c476f... 2025-01  Wiltshire Police   
                                                   2025-01  Wiltshire Police   
44e863c5fc28f43b3d8b9bac3d01cc4a98e5a628a2583e8... 2025-08  Wiltshire Police   
                                                   2025-08  Wiltshire Police   
7d209a3bc68a62a95396e05cc9b3dbf603d9a9abbaae897... 2025-08  Wiltshire Police   
                                                   2025-08  Wiltshire Police   
8786dc0a4a0c6f81c791d7cc350f98a7c864ce39a62b771... 2025-10  Wiltshire Police   
                                                   2025-10  Wiltshire Police   
de4b13b02ce9e30c0d6a494ca3b95c4d72d32e2e26297e7... 2025-08  Wiltshire Police   
                                                   2025-08  Wiltshire Police   

                                                                              Crime type  \
Crime ID                                           Month                                   
2083bb0bc793766a6c5df56889ee37e2ff5a3c7c18c476f... 2025-01     Criminal damage and arson   
                                                   2025-01                 Vehicle crime   
44e863c5fc28f43b3d8b9bac3d01cc4a98e5a628a2583e8... 2025-08                  Public order   
                                                   2025-08  Violence and sexual offences   
7d209a3bc68a62a95396e05cc9b3dbf603d9a9abbaae897... 2025-08     Criminal damage and arson   
                                                   2025-08                  Public order   
8786dc0a4a0c6f81c791d7cc350f98a7c864ce39a62b771... 2025-10                   Shoplifting   
                                                   2025-10  Violence and sexual offences   
de4b13b02ce9e30c0d6a494ca3b95c4d72d32e2e26297e7... 2025-08                   Shoplifting   
                                                   2025-08  Violence and sexual offences   

                                                                                    Last outcome category  
Crime ID                                           Month                                                   
2083bb0bc793766a6c5df56889ee37e2ff5a3c7c18c476f... 2025-01                      Status update unavailable  
                                                   2025-01  Investigation complete; no suspect identified  
44e863c5fc28f43b3d8b9bac3d01cc4a98e5a628a2583e8... 2025-08                      Status update unavailable  
                                                   2025-08                    Unable to prosecute suspect  
7d209a3bc68a62a95396e05cc9b3dbf603d9a9abbaae897... 2025-08                      Status update unavailable  
                                                   2025-08  Investigation complete; no suspect identified  
8786dc0a4a0c6f81c791d7cc350f98a7c864ce39a62b771... 2025-10                      Status update unavailable  
                                                   2025-10  Investigation complete; no suspect identified  
de4b13b02ce9e30c0d6a494ca3b95c4d72d32e2e26297e7... 2025-08                      Status update unavailable  
                                                   2025-08  Investigation complete; no suspect identified

In [17]:
crime_df[crime_df["Crime ID"].notna()].groupby(["Crime ID", "Month"])["Reported by"].nunique().value_counts()

Reported by
1    345110
Name: count, dtype: int64

### Duplicate Record Findings

The dataset contains 25,161 identical rows of which 25,145 are antisocial behaviour (ASB). ASB records have no unique Crime IDs so identical rows cannot reliably be identified as the same incident. These records are retained to avoid potentially undercounting ASB.

For recorded crimes repeated Crime IDs were investigated using the month and reporting police force. Five Crime ID/month combinations contained conflicting crime categories and were excluded because their correct classification could not be established. The remaining repeated crimes were reduced to one record per Crime ID, month and police force.

Cleaning removed 579 records, leaving 436,692.



## 3. Data Cleaning and Transformation

The findings from the data quality assessment are used to clean the dataset. ASB records are retained, records with conflicting crime categories are excluded and repeated identified crimes are reduced to one record per Crime ID, month and reporting force.

In [18]:
# Separate ASB incidents from recorded crimes
asb = crime_df[crime_df["Crime type"] == "Anti-social behaviour"].copy()
crimes = crime_df[crime_df["Crime type"] != "Anti-social behaviour"].copy()

# Exclude Crime ID/month combinations with conflicting categories
crimes = crimes[~crimes.set_index(["Crime ID", "Month"]).index.isin(conflicting_ids)].copy()

# Retain one record per identified crime, force and month
crimes = crimes.drop_duplicates(subset=["Crime ID", "Month", "Reported by"])

cleaned_df = pd.concat([crimes, asb], ignore_index=True)

Original records: 437,271
Cleaned records: 436,692
Records removed: 579


In [19]:
# Check no duplicate identified crimes remain
remaining_duplicates = cleaned_df[cleaned_df["Crime ID"].notna()].duplicated(subset=["Crime ID", "Month", "Reported by"]).sum()

assert remaining_duplicates == 0

# Check that all ASB records were retained
original_asb = (crime_df["Crime type"] == "Anti-social behaviour").sum()
cleaned_asb = (cleaned_df["Crime type"] == "Anti-social behaviour").sum()

assert original_asb == cleaned_asb

removed_records = len(crime_df) - len(cleaned_df)

assert removed_records >= 0

print(f"Original records: {len(crime_df):,}")
print(f"Cleaned records: {len(cleaned_df):,}")
print(f"Records removed: {removed_records:,}")
print("Cleaning validation passed.")

Original records: 437,271
Cleaned records: 436,692
Records removed: 579
Cleaning validation passed.


### Cleaning Validation

The checks confirm that no repeated combinations of Crime ID, month and reporting force remain, and that all ASB records have been retained.

The original dataset remains available separately as `crime_df`, allowing the cleaned records to be compared against the source data.

## 4. Aggregation for Reporting

The cleaned records are grouped by police force, month and crime category to calculate incident counts.

This provides an initial reporting dataset that can be used to check whether the later geographical aggregation preserves the same totals.

In [20]:
# Count incidents by police force, month and crime category
reporting_df = (cleaned_df.groupby(["Reported by", "Month", "Crime type"]).size().reset_index(name="Crime count"))
reporting_df.head(10)

,Reported by,Month,Crime type,Crime count
0,Dorset Police,2024-01,Anti-social behaviour,958
1,Dorset Police,2024-01,Bicycle theft,52
2,Dorset Police,2024-01,Burglary,211
3,Dorset Police,2024-01,Criminal damage and arson,417
4,Dorset Police,2024-01,Drugs,80
5,Dorset Police,2024-01,Other crime,43
6,Dorset Police,2024-01,Other theft,336
7,Dorset Police,2024-01,Possession of weapons,42
8,Dorset Police,2024-01,Public order,262
9,Dorset Police,2024-01,Robbery,21


In [21]:
# # Check each force/month/category combination is unique
assert not reporting_df.duplicated(subset=["Reported by", "Month", "Crime type"]).any()

# Check total incidents match the cleaned dataset
assert reporting_df["Crime count"].sum() == len(cleaned_df)

# Check force and month coverage
assert reporting_df["Reported by"].nunique() == len(forces)
assert reporting_df["Month"].nunique() == len(months)

# Summarise results
print("Reporting rows:", len(reporting_df))
print("Total inncidents:", reporting_df["Crime count"].sum())
print("Police forces:", reporting_df["Reported by"].nunique())
print("Months:", reporting_df["Month"].nunique())
print("Force aggregation validated.")

Reporting rows: 1344
Total inncidents: 436692
Police forces: 4
Months: 24
Force aggregation validated.


### Aggregation Results

The initial aggregation produced 1,344 force level reporting rows from 436,692 cleaned records.

This is an intermediate dataset used to validate the later geographical aggregation.Adding LSOA level detail increases the number of reporting rows but the total incident count must remain unchanged.

Antisocial behaviour remains included and is distinguished from recorded crime in the final dataset..

## 5. Population Enrichment
ONS mid-year population estimates for 2024 and 2025 are used to calculate population adjusted crime rates allowing comparisons between police forces of different sizes. The source workbook is inspected first to identify its structure and column headings before extracting the required population figures.

In [22]:
# Check available worksheets
population_file = "enrichment_data/policeforceareasmid2011tomid2025.xlsx"

excel_file = pd.ExcelFile(population_file)

print(excel_file.sheet_names)

['Cover sheet', 'Contents', 'Notes', 'Related publications', 'Mid-2011 to Mid-2025']


In [23]:
# Inspect the worksheet to identify the header row
population_sample = pd.read_excel(population_file, sheet_name="Mid-2011 to Mid-2025", header=None, nrows=15)
population_sample

,0,1,2,3,4,5,6,7,8,9,...,175,176,177,178,179,180,181,182,183,184
0,Estimates by single year of age and sex for 20...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,This worksheet contains one table.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,To turn off freeze panes select the 'View' rib...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PFA 2023 Code,PFA 2023 Name,Mid-Year,F0,F1,F2,F3,F4,F5,F6,...,M81,M82,M83,M84,M85,M86,M87,M88,M89,M90
4,E23000001,Metropolitan Police,mid-2011,61350,59379,57697,57186,54527,52518,49086,...,12373,10812,9522,8565,7570,6530,5780,4857,4163,11990
5,E23000001,Metropolitan Police,mid-2012,64693,60570,58582,57211,56859,54303,52219,...,12387,11448,9909,8637,7738,6729,5760,5043,4158,12864
6,E23000001,Metropolitan Police,mid-2013,63201,63403,59518,57938,56781,56590,53985,...,12591,11369,10485,9005,7787,6863,5890,5002,4282,13532
7,E23000001,Metropolitan Police,mid-2014,61334,61617,61625,58383,57221,56382,56211,...,12422,11739,10384,9581,8164,7033,6100,5150,4271,14536
8,E23000001,Metropolitan Police,mid-2015,61691,59485,59715,60000,57438,56610,55980,...,12501,11507,10822,9375,8582,7227,6135,5266,4306,14827
9,E23000001,Metropolitan Police,mid-2016,62053,59758,57636,58046,58540,56678,55992,...,12699,11599,10660,9883,8389,7631,6331,5386,4471,15422


In [24]:
# The fourth row contains the column headings
population_df = pd.read_excel(population_file, sheet_name="Mid-2011 to Mid-2025", header=3)
population_df.head(5)

,PFA 2023 Code,PFA 2023 Name,Mid-Year,F0,F1,F2,F3,F4,F5,F6,...,M81,M82,M83,M84,M85,M86,M87,M88,M89,M90
0,E23000001,Metropolitan Police,mid-2011,61350,59379,57697,57186,54527,52518,49086,...,12373,10812,9522,8565,7570,6530,5780,4857,4163,11990
1,E23000001,Metropolitan Police,mid-2012,64693,60570,58582,57211,56859,54303,52219,...,12387,11448,9909,8637,7738,6729,5760,5043,4158,12864
2,E23000001,Metropolitan Police,mid-2013,63201,63403,59518,57938,56781,56590,53985,...,12591,11369,10485,9005,7787,6863,5890,5002,4282,13532
3,E23000001,Metropolitan Police,mid-2014,61334,61617,61625,58383,57221,56382,56211,...,12422,11739,10384,9581,8164,7033,6100,5150,4271,14536
4,E23000001,Metropolitan Police,mid-2015,61691,59485,59715,60000,57438,56610,55980,...,12501,11507,10822,9375,8582,7227,6135,5266,4306,14827


In [25]:
# Inspect the age and sex columns
print(population_df.columns.tolist())

['PFA 2023 Code', 'PFA 2023 Name', 'Mid-Year', 'F0', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28', 'F29', 'F30', 'F31', 'F32', 'F33', 'F34', 'F35', 'F36', 'F37', 'F38', 'F39', 'F40', 'F41', 'F42', 'F43', 'F44', 'F45', 'F46', 'F47', 'F48', 'F49', 'F50', 'F51', 'F52', 'F53', 'F54', 'F55', 'F56', 'F57', 'F58', 'F59', 'F60', 'F61', 'F62', 'F63', 'F64', 'F65', 'F66', 'F67', 'F68', 'F69', 'F70', 'F71', 'F72', 'F73', 'F74', 'F75', 'F76', 'F77', 'F78', 'F79', 'F80', 'F81', 'F82', 'F83', 'F84', 'F85', 'F86', 'F87', 'F88', 'F89', 'F90', 'M0', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M19', 'M20', 'M21', 'M22', 'M23', 'M24', 'M25', 'M26', 'M27', 'M28', 'M29', 'M30', 'M31', 'M32', 'M33', 'M34', 'M35', 'M36', 'M37', 'M38', 'M39', 'M40', 'M41', 'M42', 'M43', 'M44', 'M45', 'M46', 'M47', 


### Calculating Total Population

The ONS dataset separates population estimates by age and sex. These columns are summed to calculate the total population of each police force area for each year.

In [26]:
# Identify all age and sex columns
age_columns = [col for col in population_df.columns if col.startswith(("F", "M")) and col[1:].isdigit()]

# Sum population across all ages and both sexes
population_df["Population"] = population_df[age_columns].sum(axis=1)

# Check result 
population_df[["PFA 2023 Name", "Mid-Year", "Population"]].head()

,PFA 2023 Name,Mid-Year,Population
0,Metropolitan Police,mid-2011,8196995
1,Metropolitan Police,mid-2012,8313253
2,Metropolitan Police,mid-2013,8431450
3,Metropolitan Police,mid-2014,8539625
4,Metropolitan Police,mid-2015,8651877


In [27]:
population_df["PFA 2023 Name"].unique()

array(['Metropolitan Police', 'Cumbria', 'Lancashire', 'Merseyside',
       'Greater Manchester', 'Cheshire', 'Northumbria', 'Durham',
       'North Yorkshire', 'West Yorkshire', 'South Yorkshire',
       'Humberside', 'Cleveland', 'West Midlands', 'Staffordshire',
       'West Mercia', 'Warwickshire', 'Derbyshire', 'Nottinghamshire',
       'Lincolnshire', 'Leicestershire', 'Northamptonshire',
       'Cambridgeshire', 'Norfolk', 'Suffolk', 'Bedfordshire',
       'Hertfordshire', 'Essex', 'Thames Valley', 'Hampshire', 'Surrey',
       'Kent', 'Sussex', 'London, City of', 'Devon & Cornwall',
       'Avon and Somerset', 'Gloucestershire', 'Wiltshire', 'Dorset',
       'North Wales', 'Gwent', 'South Wales', 'Dyfed-Powys'], dtype=object)

### Matching Population to Crime Data

The relevant police forces and reporting years are selected from the ONS dataset. Police force names are matched to those used in the crime data.

Population estimates are annual so the year is extracted from each reporting month to join the appropriate population figure to the monthly crime counts.

In [28]:
# Convert filename identifiers to ONS naming
ons_forces = [force.title() for force in forces]

population_clean = population_df[population_df["PFA 2023 Name"].isin(ons_forces) & population_df["Mid-Year"].isin(["mid-2024", "mid-2025"])][["PFA 2023 Name", "Mid-Year", "Population"]].copy()

population_clean

,PFA 2023 Name,Mid-Year,Population
253,Warwickshire,mid-2024,633002
254,Warwickshire,mid-2025,644021
553,Gloucestershire,mid-2024,668109
554,Gloucestershire,mid-2025,671908
568,Wiltshire,mid-2024,767325
569,Wiltshire,mid-2025,771678
583,Dorset,mid-2024,797350
584,Dorset,mid-2025,796865


In [29]:
# Match ONS force names to crime dataset
force_mapping = {
    "Dorset": "Dorset Police",
    "Wiltshire": "Wiltshire Police",
    "Gloucestershire": "Gloucestershire Constabulary",
    "Warwickshire": "Warwickshire Police"}

population_clean["Reported by"] = (population_clean["PFA 2023 Name"].map(force_mapping))

# Extract the year from values like"mid-2024"
population_clean["Year"] = (population_clean["Mid-Year"].str[-4:].astype(int))

# Extract the year from the monthly crime data
reporting_df["Year"] = (reporting_df["Month"].str[:4].astype(int))
population_clean[["Reported by", "Year", "Population"]]

,Reported by,Year,Population
253,Warwickshire Police,2024,633002
254,Warwickshire Police,2025,644021
553,Gloucestershire Constabulary,2024,668109
554,Gloucestershire Constabulary,2025,671908
568,Wiltshire Police,2024,767325
569,Wiltshire Police,2025,771678
583,Dorset Police,2024,797350
584,Dorset Police,2025,796865


In [30]:
# Identify years covered by the crime data
crime_years = set(reporting_df["Month"].str[:4].astype(int))

# Expected force/year combinations
expected_pairs = {(force, year) for force in force_mapping.values()for year in crime_years}

# Actual force/year combinations in the population data
actual_pairs = set(zip(population_clean["Reported by"], population_clean["Year"]))

# Check that population data covers every required force and year
assert actual_pairs == expected_pairs, ( f"Missing: {expected_pairs - actual_pairs}; "f"Unexpected: {actual_pairs - expected_pairs}")

# Check for duplicates and invalid values
assert not population_clean.duplicated(subset=["Reported by", "Year"]).any()
assert population_clean["Population"].notna().all()
assert population_clean["Population"].gt(0).all()


print("Population rows:",len(population_clean))
print("Population coverage validated.")

Population rows: 8
Population coverage validated.


In [31]:
# Join annual population to monthly crime counts 
enriched_df = reporting_df.merge(
    population_clean[["Reported by", "Year", "Population"]],
    on=["Reported by", "Year"],
    how="left",
    validate="many_to_one",
    indicator=True)

print(enriched_df["_merge"].value_counts())
print("Rows before:", len(reporting_df))
print("Rows after:", len(enriched_df))

_merge
both          1344
left_only        0
right_only       0
Name: count, dtype: int64
Rows before: 1344
Rows after: 1344


In [32]:
# Check that all rows matched without changing the row count
assert (enriched_df["_merge"] == "both").all()
assert len(enriched_df) == len(reporting_df)
assert enriched_df["Population"].notna().all()
assert enriched_df["Population"].gt(0).all()

enriched_df = enriched_df.drop(columns="_merge")

In [33]:
# rename count to reflect inclusion of ASB incidents
enriched_df = enriched_df.rename(columns={"Crime count": "Incident count"})

# Distinguish recorded crimes from ASB incidents
enriched_df["Incident type"] = enriched_df["Crime type"].apply(lambda x: "ASB" if x == "Anti-social behaviour" else "Recorded crime")

# Calculate monthly incidents per 1,000 residents
enriched_df["Rate per 1000"] = (enriched_df["Incident count"] / enriched_df["Population"] * 1000).round(2)

enriched_df.head()

,Reported by,Month,Crime type,Incident count,Year,Population,Incident type,Rate per 1000
0,Dorset Police,2024-01,Anti-social behaviour,958,2024,797350,ASB,1.20
1,Dorset Police,2024-01,Bicycle theft,52,2024,797350,Recorded crime,0.07
2,Dorset Police,2024-01,Burglary,211,2024,797350,Recorded crime,0.26
3,Dorset Police,2024-01,Criminal damage and arson,417,2024,797350,Recorded crime,0.52
4,Dorset Police,2024-01,Drugs,80,2024,797350,Recorded crime,0.10


In [34]:
#Validate calculated rates
assert enriched_df["Rate per 1000"].notna().all()
assert enriched_df["Rate per 1000"].ge(0).all()

assert enriched_df["Incident count"].sum() == len(cleaned_df)

assert not enriched_df.duplicated(subset=["Reported by", "Month", "Crime type"]).any()

print("Reporting rows:", len(enriched_df))
print("Total incidents:", enriched_df["Incident count"].sum())

Reporting rows: 1344
Total incidents: 436692


### Population Enrichment Results

All 1,344 reporting rows were successfully matched to population estimates without introducing duplicates.

Monthly incident rates per 1,000 residents were calculated using the population estimate for the corresponding year. Annual mid year estimates are applied to every month within that year, so population changes within the year are not reflected.

Antisocial behaviour is identified separately from recorded crime to allow more accurate comparisons.

## 6. Socioeconomic Enrichment
The English Indices of Deprivation 2025 dataset provides socioeconomic context for comparing crime across police force areas.

Income deprivation was selected instead of the overall Index of Multiple Deprivation (IMD), which includes crime as one of its measures. Using overall IMD could distort the relationship being investigated because crime would already contribute to the deprivation score.

The income deprivation scores are combined using population weighted averages to produce one figure per police force area.


In [35]:
deprivation_file = "enrichment_data/File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv"

deprivation_df = pd.read_csv(deprivation_file)

print(deprivation_df.shape)

deprivation_df[["LSOA code (2021)","Local Authority District name (2024)", "Income Score (rate)", "Total population: mid 2022"]].head()

(33755, 56)


,LSOA code (2021),Local Authority District name (2024),Income Score (rate),Total population: mid 2022
0,E01000001,City of London,0.013,1795
1,E01000002,City of London,0.018,1671
2,E01000003,City of London,0.107,1896
3,E01000005,City of London,0.211,1737
4,E01000006,Barking and Dagenham,0.343,1837


In [36]:
deprivation_df[deprivation_df["Local Authority District name (2024)"].str.contains("Dorset|Bournemouth|Poole|Wiltshire|Gloucester|Warwick",case=False,na=False)]["Local Authority District name (2024)"].unique()

array(['South Gloucestershire', 'Bournemouth, Christchurch and Poole',
       'Dorset', 'Gloucester', 'North Warwickshire', 'Warwick',
       'Wiltshire'], dtype=object)

### Matching Local Authorities to Police Forces

The deprivation dataset identifies areas using Local Authority District (LAD) codes rather than police force names.

A geographical lookup is used to identify which local authorities belong to each police force. This allows the LSOAs to be selected and grouped correctly.

In [37]:
lookup_df = pd.read_excel("enrichment_data/lookuptablefortheukauthoritycodesfor2024.xlsx")

lookup_df[["LAD24CD", "LAD24NM", "PFA24NM"]].head()

,LAD24CD,LAD24NM,PFA24NM
0,E06000001,Hartlepool,Cleveland
1,E06000002,Middlesbrough,Cleveland
2,E06000003,Redcar and Cleveland,Cleveland
3,E06000004,Stockton-on-Tees,Cleveland
4,E06000005,Darlington,Durham


In [38]:
selected_forces = ons_forces

force_lookup = lookup_df[lookup_df["PFA24NM"].isin(selected_forces)][["LAD24CD", "LAD24NM", "PFA24NM"]].drop_duplicates()

print(force_lookup.groupby("PFA24NM").size())
print("Duplicate local authority codes:", force_lookup["LAD24CD"].duplicated().sum())

force_lookup

PFA24NM
Dorset             2
Gloucestershire    6
Warwickshire       5
Wiltshire          2
dtype: int64
Duplicate local authority codes: 0


,LAD24CD,LAD24NM,PFA24NM
27,E06000030,Swindon,Wiltshire
50,E06000054,Wiltshire,Wiltshire
54,E06000058,"Bournemouth, Christchurch and Poole",Dorset
57,E06000059,Dorset,Dorset
114,E07000078,Cheltenham,Gloucestershire
115,E07000079,Cotswold,Gloucestershire
116,E07000080,Forest of Dean,Gloucestershire
117,E07000081,Gloucester,Gloucestershire
118,E07000082,Stroud,Gloucestershire
119,E07000083,Tewkesbury,Gloucestershire


In [39]:
[col for col in deprivation_df.columns if "Local Authority" in col]

['Local Authority District code (2024)',
 'Local Authority District name (2024)']

In [40]:
#Join the deprivation data to the lookup

deprivation_joined = deprivation_df.merge(
    force_lookup,
    left_on="Local Authority District code (2024)",
    right_on="LAD24CD",
    how="inner",
    validate="many_to_one")

print("Matched LSOAs:", len(deprivation_joined))
print(deprivation_joined["PFA24NM"].value_counts())

Matched LSOAs: 1657
PFA24NM
Dorset             463
Wiltshire          447
Gloucestershire    393
Warwickshire       354
Name: count, dtype: int64


In [41]:
# Validate the join

print("Missing income scores:", deprivation_joined["Income Score (rate)"].isna().sum())
print("Missing population denominators:", deprivation_joined["Total population: mid 2022"].isna().sum())
print("Local authorities matched:", deprivation_joined["LAD24CD"].nunique())
print("Police forces matched:", deprivation_joined["PFA24NM"].nunique())

Missing income scores: 0
Missing population denominators: 0
Local authorities matched: 15
Police forces matched: 4



### Calculating Force-Level Deprivation

Each LSOA's income deprivation score is weighted by its mid 2022 population. The weighted scores are then combined to calculate an average for each police force area.

This gives more weight to areas with larger populations rather than treating every LSOA equally.


In [42]:
# Ensure population denominators are valid
assert deprivation_joined["Total population: mid 2022"].gt(0).all()

# Check that each LSOA appears only once
assert not deprivation_joined["LSOA code (2021)"].duplicated().any()

# Check all selected local authorities are represented
assert set(force_lookup["LAD24CD"]) == set(deprivation_joined["LAD24CD"])

# Weight each score by its LSOA population
deprivation_joined["Weighted income"] = (deprivation_joined["Income Score (rate)"] * deprivation_joined["Total population: mid 2022"])

# Aggregate to police force level
force_deprivation = deprivation_joined.groupby("PFA24NM", as_index=False).agg(
    weighted_income=("Weighted income", "sum"),
    total_population=("Total population: mid 2022", "sum"))

force_deprivation["Income deprivation rate"] = (force_deprivation["weighted_income"] / force_deprivation["total_population"])

force_deprivation = force_deprivation[["PFA24NM", "Income deprivation rate"]]

force_deprivation

,PFA24NM,Income deprivation rate
0,Dorset,0.181604
1,Gloucestershire,0.164122
2,Warwickshire,0.169107
3,Wiltshire,0.164614


In [43]:
# Check that each force has one valid deprivation rate

assert set(force_deprivation["PFA24NM"]) == set(selected_forces)
assert force_deprivation["Income deprivation rate"].between(0, 1).all()
assert force_deprivation["PFA24NM"].is_unique

In [44]:
# Add deprivation rates to the monthly reporting data

force_deprivation["Reported by"] = (force_deprivation["PFA24NM"].map(force_mapping))

final_df = enriched_df.merge(
    force_deprivation[["Reported by", "Income deprivation rate"]
    ],
    on="Reported by",
    how="left",
    validate="many_to_one",
    indicator=True)

print(final_df["_merge"].value_counts())
print("Rows before:", len(enriched_df))
print("Rows after:", len(final_df))

_merge
both          1344
left_only        0
right_only       0
Name: count, dtype: int64
Rows before: 1344
Rows after: 1344


In [45]:
# Check all rows matched and no additional rows were introduced

assert (final_df["_merge"] == "both").all()
assert len(final_df) == len(enriched_df)
assert final_df["Income deprivation rate"].notna().all()

assert not final_df.duplicated(subset=["Reported by", "Month", "Crime type"]).any()

final_df = final_df.drop(columns="_merge")
final_df.head()

,Reported by,Month,Crime type,Incident count,Year,Population,Incident type,Rate per 1000,Income deprivation rate
0,Dorset Police,2024-01,Anti-social behaviour,958,2024,797350,ASB,1.20,0.181604
1,Dorset Police,2024-01,Bicycle theft,52,2024,797350,Recorded crime,0.07,0.181604
2,Dorset Police,2024-01,Burglary,211,2024,797350,Recorded crime,0.26,0.181604
3,Dorset Police,2024-01,Criminal damage and arson,417,2024,797350,Recorded crime,0.52,0.181604
4,Dorset Police,2024-01,Drugs,80,2024,797350,Recorded crime,0.10,0.181604


In [46]:
# standardise reporting column names
final_df = final_df.rename(columns={
    "Reported by": "police_force",
    "Month": "month",
    "Crime type": "crime_category",
    "Incident count": "incident_count",
    "Year": "year",
    "Population": "population",
    "Incident type": "incident_type",
    "Rate per 1000": "rate_per_1000",
    "Income deprivation rate": "income_deprivation_rate"})

# Recalculate monthly rates from incident counts and population
final_df["rate_per_1000"] = (final_df["incident_count"] / final_df["population"] * 1000).round(4)

In [47]:
# Check the force level dataset before geographical enrichment

# Check that required fields contain no missing values
assert final_df.notna().all().all()

# Ensure one row per reporting combination
assert not final_df.duplicated(subset=["police_force", "month", "crime_category"]).any()

# # Check that incident totals match the cleaned data
assert final_df["incident_count"].sum() == len(cleaned_df)

# Check expected reporting coverage
assert final_df["police_force"].nunique() == len(forces)
assert set(final_df["month"]) == set(months.astype(str))

# Confirm valid values
assert final_df["population"].gt(0).all()
assert final_df["incident_count"].ge(0).all()
assert final_df["income_deprivation_rate"].between(0, 1).all()

print("Force level validation checks passed.")
print("Force level dataset shape:", final_df.shape)

Force level validation checks passed.
Force level dataset shape: (1344, 9)


### Socioeconomic Enrichment Results

All 1,344 reporting rows were successfully matched to a force level income deprivation rate without introducing duplicates or changing incident totals.

The 2025 deprivation figures are used as contextual information for both 2024 and 2025 crime data. They do not measure changes in deprivation between those years and any relationship with crime should not be interpreted as evidence of causation.

## 7. Geographical Aggregation and Enrichment 

Police.uk provides approximate incident locations, coordinates and Lower Layer Super Output Area (LSOA) codes.

LSOAs were selected for geographical reporting because they provide consistent areas that can be matched to population and deprivation data.

Individual street locations and coordinates are excluded from the final dataset. Police.uk locations are anonymised and should not be treated as exact incident locations.

### 7.1 Geographical Data Quality Assessment
Crime LSOA codes are compared with the English deprivation dataset to identify which records can be matched to local population and deprivation information.

Missing, unmatched and conflicting locations are investigated separately to avoid excluding incidents unnecessarily.

In [48]:

# Compare crime LSOA codes with the deprivation dataset

crime_lsoas = set(cleaned_df["LSOA code"].dropna().unique())

population_lsoas = set(deprivation_df["LSOA code (2021)"].dropna().unique())

matching = crime_lsoas & population_lsoas

print("Crime LSOAs:", len(crime_lsoas))
print("Population LSOAs:", len(population_lsoas))
print("Matching codes:", len(matching))
print("Unmatched crime codes:", len(crime_lsoas - population_lsoas))

Crime LSOAs: 2593
Population LSOAs: 33755
Matching codes: 2534
Unmatched crime codes: 59


In [49]:
# Count records with unmatched LSOA codes
unmatched_records = cleaned_df[cleaned_df["LSOA code"].isin(crime_lsoas - population_lsoas)]

print("Records in unmatched LSOAs:", len(unmatched_records))
print("Percentage of cleaned records:", round(len(unmatched_records) / len(cleaned_df) * 100, 2))
print(unmatched_records["Reported by"].value_counts())

# 4694/431,705 records with no loaction data
# Final report use Matched/Welsh/LSOA/Unknown location? to retain all records but allow filtering

Records in unmatched LSOAs: 293
Percentage of cleaned records: 0.07
Reported by
Gloucestershire Constabulary    263
Warwickshire Police              16
Wiltshire Police                 14
Name: count, dtype: int64


In [50]:
# Inspect unmatched areas and their reporting forces
unmatched_records[["LSOA code", "LSOA name", "Reported by"]].drop_duplicates().head(20)

,LSOA code,LSOA name,Reported by
3844,W01000728,Carmarthenshire 003D,Gloucestershire Constabulary
3845,W01000735,Carmarthenshire 027D,Gloucestershire Constabulary
6919,W01000040,Isle of Anglesey 004D,Gloucestershire Constabulary
6924,W01000574,Pembrokeshire 015A,Gloucestershire Constabulary
11484,W01001556,Monmouthshire 006B,Wiltshire Police
20909,W01000031,Isle of Anglesey 001E,Gloucestershire Constabulary
20910,W01000037,Isle of Anglesey 003G,Gloucestershire Constabulary
20926,W01001897,Powys 020D,Gloucestershire Constabulary
25251,W01001619,Newport 003C,Wiltshire Police
35036,W01000004,Isle of Anglesey 001B,Gloucestershire Constabulary


In [51]:
# Check whether unmatched codes are Welsh LSOAs
unmatched_records["LSOA code"].str[0].value_counts()

LSOA code
W    293
Name: count, dtype: int64

In [52]:
#deprivation dataset and crime data set use same LSOA codes
[col for col in deprivation_df.columns if "LSOA" in col]

['LSOA code (2021)',
 'LSOA name (2021)',
 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)',
 'Income Decile (where 1 is most deprived 10% of LSOAs)',
 'Employment Decile (where 1 is most deprived 10% of LSOAs)',
 'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)',
 'Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs)',
 'Crime Decile (where 1 is most deprived 10% of LSOAs)',
 'Barriers to Housing and Services Decile (where 1 is most deprived 10% of LSOAs)',
 'Living Environment Decile (where 1 is most deprived 10% of LSOAs)',
 'Income Deprivation Affecting Children Index (IDACI) Decile (where 1 is most deprived 10% of LSOAs)',
 'Income Deprivation Affecting Older People (IDAOPI) Decile (where 1 is most deprived 10% of LSOAs)',
 'Adult Skills Sub-domain Decile (where 1 is most deprived 10% of LSOAs)',
 'Geographical Barriers Sub-domain Decile (where 1 is most deprived 10% of LSOAs)',
 'Wider B

In [53]:
# Check same code also has the same LSOA name in both datasets
name_check = (cleaned_df[["LSOA code", "LSOA name"]].drop_duplicates().merge(
    deprivation_df[["LSOA code (2021)", "LSOA name (2021)"]],
    left_on="LSOA code",
    right_on="LSOA code (2021)",
    how="inner",
    validate="many_to_one"))

name_check["names_match"] = (name_check["LSOA name"] == name_check["LSOA name (2021)"])

print(name_check["names_match"].value_counts())

#All 2534 matching English LSOA codes also have identical names

names_match
True    2534
Name: count, dtype: int64


In [54]:
# Count records without an LSOA code

missing_location = cleaned_df["LSOA code"].isna().sum()

print("Records with missing LSOA:", missing_location)
print("Percentage:", round(missing_location / len(cleaned_df) * 100, 2))

Records with missing LSOA: 4694
Percentage: 1.07


In [55]:
# Examine crimes with multiple recorded LSOAs
location_conflicts = (crime_df[crime_df["Crime ID"].notna()].groupby(["Crime ID", "Month", "Reported by"])["LSOA code"].nunique())

print("Crimes with conflicting LSOAs:", (location_conflicts > 1).sum())

Crimes with conflicting LSOAs: 322


In [56]:

conflicting_locations = location_conflicts[location_conflicts > 1].index

print("Conflicting crime combinations:", len(conflicting_locations))

Conflicting crime combinations: 322


In [57]:
# flag crimes with conflicting locations
cleaned_df["location_conflict"] = (cleaned_df.set_index(["Crime ID", "Month", "Reported by"]).index.isin(conflicting_locations))

print(cleaned_df["location_conflict"].value_counts())

location_conflict
False    436370
True        322
Name: count, dtype: int64


In [58]:
import numpy as np

cleaned_df["location_status"] = np.select([cleaned_df["location_conflict"], cleaned_df["LSOA code"].isna(), cleaned_df["LSOA code"].str.startswith("W", na=False)],
                                          ["Ambiguous location","Unknown location","Welsh LSOA"],
                                          default="Matched English LSOA")

print(cleaned_df["location_status"].value_counts())

# Check that every record has a location status
assert cleaned_df["location_status"].notna().all()
assert (cleaned_df["location_status"].value_counts().sum() == len(cleaned_df))

print("\nClassified successfully")

location_status
Matched English LSOA    431383
Unknown location          4694
Ambiguous location         322
Welsh LSOA                 293
Name: count, dtype: int64

Classified successfully


In [59]:
valid_lsoas = set(deprivation_df["LSOA code (2021)"])

invalid_matches = cleaned_df[(cleaned_df["location_status"] == "Matched English LSOA") & ~cleaned_df["LSOA code"].isin(valid_lsoas)]

assert len(invalid_matches) == 0

print("matched english LSOAs validated")

matched english LSOAs validated



### Geographical Data Quality Findings

Of the 2,593 distinct LSOA codes in the crime data, 2,534 matched the English deprivation dataset. The remaining 59 were Welsh LSOAs, accounting for 293 records.

A further 4,694 records had missing LSOA codes. Some identified crimes also had conflicting locations in the source data.

These records are retained with separate location statuses, allowing them to contribute to overall crime counts without being assigned an unreliable English location.

A small number of incidents were associated with LSOAs outside the selected police force areas. These records were retained under their reporting police force to preserve the original incident counts.


### 7.2 LSOA-Level Aggregation

Crime records are grouped by police force, month, crime category and geographical area.

Valid English LSOAs retain their codes. Records without a usable English LSOA are grouped by location status instead, preserving their incident counts without assigning them to an incorrect area.

In [60]:
# # Use valid LSOA codes, otherwise group by location status
geo_df = cleaned_df.copy()

geo_df["reporting_lsoa"] = np.where(geo_df["location_status"] == "Matched English LSOA", geo_df["LSOA code"], geo_df["location_status"])

geo_reporting = (geo_df.groupby(["Reported by", "Month", "Crime type", "reporting_lsoa", "location_status"],dropna=False).size().reset_index(name="incident_count"))

print("Geographical reporting rows:", len(geo_reporting))
print("Total incidents:", geo_reporting["incident_count"].sum())

assert geo_reporting["incident_count"].sum() == len(cleaned_df)

Geographical reporting rows: 171406
Total incidents: 436692


In [61]:
lsoa_enrichment = deprivation_df[["LSOA code (2021)", "LSOA name (2021)", "Total population: mid 2022", "Income Score (rate)"]].copy()

lsoa_enrichment = lsoa_enrichment.rename(columns={
    "LSOA code (2021)": "reporting_lsoa",
    "LSOA name (2021)": "lsoa_name",
    "Total population: mid 2022": "lsoa_population_2022",
    "Income Score (rate)": "lsoa_income_deprivation_rate"})

assert lsoa_enrichment["reporting_lsoa"].is_unique
assert lsoa_enrichment["lsoa_population_2022"].gt(0).all()
assert lsoa_enrichment["lsoa_income_deprivation_rate"].between(0, 1).all()


### 7.3 LSOA Population and Deprivation Enrichment

The geographical counts are matched to mid 2022 LSOA population estimates and 2025 income deprivation scores.

Monthly incident rates per 1,000 residents are calculated for matched English LSOAs. Records without a valid match retain their incident counts but have no local population adjusted rate.


In [62]:
# Add population and deprivation data using LSOA codes
geo_enriched = geo_reporting.merge(
    lsoa_enrichment,
    on="reporting_lsoa",
    how="left",
    validate="many_to_one",
    indicator=True)

print(geo_enriched["_merge"].value_counts())

assert len(geo_enriched) == len(geo_reporting)

assert (geo_enriched.loc[geo_enriched["location_status"] == "Matched English LSOA", "_merge"] == "both").all()

_merge
both          170569
left_only        837
right_only         0
Name: count, dtype: int64


In [63]:
# Calculate monthly incident rates per 1000 residents
geo_enriched["lsoa_rate_per_1000"] = (geo_enriched["incident_count"] / geo_enriched["lsoa_population_2022"]* 1000)

geo_enriched.loc[geo_enriched["location_status"] != "Matched English LSOA","lsoa_rate_per_1000"] = np.nan

print("geographical rows:", len(geo_enriched))
print("total incidents:", geo_enriched["incident_count"].sum())
print("Rows without local rates:", geo_enriched["lsoa_rate_per_1000"].isna().sum())

geographical rows: 171406
total incidents: 436692
Rows without local rates: 837


In [64]:
# Check geographical incident totals against the initial aggregation
geo_totals = (geo_enriched.groupby(["Reported by", "Month", "Crime type"])["incident_count"].sum().sort_index())

original_totals = (reporting_df.groupby(["Reported by", "Month", "Crime type"])["Crime count"].sum().sort_index())

pd.testing.assert_series_equal(geo_totals, original_totals, check_names=False)
print("Geographical totals match OG reporting totals.")

Geographical totals match OG reporting totals.


### 7.4 Force-Level Enrichment

Annual police force population and income deprivation figures are added to the geographical dataset.

The join uses police force and year, with checks to confirm that all records were matched and no additional rows were introduced.

In [65]:
force_enrichment = final_df[["police_force", "year","population","income_deprivation_rate"]].drop_duplicates()

# 1 record per police force and year
assert not force_enrichment.duplicated(subset=["police_force", "year"]).any()

print(force_enrichment.shape)

(8, 4)


In [66]:
geo_enriched["year"] = geo_enriched["Month"].str[:4].astype(int)

force_enrichment = force_enrichment.rename(columns={"population": "force_population","income_deprivation_rate": "force_income_deprivation_rate"})

In [67]:
geo_final = geo_enriched.merge(
    force_enrichment,
    left_on=["Reported by", "year"],
    right_on=["police_force", "year"],
    how="left",
    validate="many_to_one",
    indicator="force_merge")

print(geo_final["force_merge"].value_counts())
print("rows before:", len(geo_enriched))
print("rows after:", len(geo_final))

force_merge
both          171406
left_only          0
right_only         0
Name: count, dtype: int64
rows before: 171406
rows after: 171406


In [68]:
# Check the join and confirm incident totals have not changed
assert (geo_final["force_merge"] == "both").all()
assert len(geo_final) == len(geo_enriched)
assert geo_final["force_population"].gt(0).all()
assert geo_final["force_income_deprivation_rate"].between(0, 1).all()
assert geo_final["incident_count"].sum() == len(cleaned_df)

print("Validated.")

Validated.


In [69]:
geo_final = geo_final.rename(columns={"Month": "month", "Crime type": "crime_category", "lsoa_rate_per_1000": "lsoa_monthly_rate_per_1000"})

# Leave LSOA codes blank for unmatched locations 
geo_final["lsoa_code"] = geo_final["reporting_lsoa"].where(geo_final["location_status"] == "Matched English LSOA")

geo_final["incident_type"] = np.where(geo_final["crime_category"] == "Anti-social behaviour","ASB","Recorded crime")

reporting_columns = [
    "police_force",
    "year",
    "month",
    "crime_category",
    "incident_type",
    "location_status",
    "lsoa_code",
    "lsoa_name",
    "incident_count",
    "lsoa_population_2022",
    "lsoa_income_deprivation_rate",
    "lsoa_monthly_rate_per_1000",
    "force_population",
    "force_income_deprivation_rate"]

geo_final_csv = geo_final[reporting_columns].copy()
geo_final_csv.head()

,police_force,year,month,crime_category,incident_type,location_status,lsoa_code,lsoa_name,incident_count,lsoa_population_2022,lsoa_income_deprivation_rate,lsoa_monthly_rate_per_1000,force_population,force_income_deprivation_rate
0,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015273,"Bournemouth, Christchurch and Poole 035B",2,2118.0,0.315,0.944287,797350,0.181604
1,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015274,"Bournemouth, Christchurch and Poole 036A",3,1844.0,0.179,1.626898,797350,0.181604
2,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015275,"Bournemouth, Christchurch and Poole 035C",3,1704.0,0.189,1.760563,797350,0.181604
3,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015276,"Bournemouth, Christchurch and Poole 035D",2,1769.0,0.229,1.130582,797350,0.181604
4,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015277,"Bournemouth, Christchurch and Poole 035E",7,2028.0,0.312,3.451677,797350,0.181604



### 7.5 Ward Lookup and Geographical Features

An ONS geographical lookup is used to match 2021 LSOAs to 2022 electoral wards, providing more recognisable geographical names for reporting.

The lookup uses a best fit match, meaning an LSOA may not fall entirely within its assigned ward.

Local authority names and shorter LSOA identifiers are also extracted to make the data easier to use in Power BI.

In [70]:
ward_file = "enrichment_data/LSOA_(2021)_to_Ward_to_Local_Authority_District_(May_2022)_Best_Fit_Lookup_for_EW.csv"
ward_df = pd.read_csv(ward_file)

print(ward_df.shape)
print(ward_df.columns.tolist())
ward_df.head()

# Inspect the available LSOA and ward fields

(35672, 9)
['ObjectId', 'LSOA21CD', 'LSOA21NM', 'WD22CD', 'WD22NM', 'WD22NMW', 'LTLA22CD', 'LTLA22NM', 'LTLA22NMW']


,ObjectId,LSOA21CD,LSOA21NM,WD22CD,WD22NM,WD22NMW,LTLA22CD,LTLA22NM,LTLA22NMW
0,1,E01000001,City of London 001A,E05009288,Aldersgate,NaN,E09000001,City of London,NaN
1,2,E01000002,City of London 001B,E05009302,Cripplegate,NaN,E09000001,City of London,NaN
2,3,E01000003,City of London 001C,E05009302,Cripplegate,NaN,E09000001,City of London,NaN
3,4,E01000005,City of London 001E,E05009308,Portsoken,NaN,E09000001,City of London,NaN
4,5,E01000006,Barking and Dagenham 016A,E05014066,Northbury,NaN,E09000002,Barking and Dagenham,NaN


In [71]:
ward_df[ward_df["LTLA22NM"] == "Bournemouth, Christchurch and Poole"][["LSOA21NM", "WD22NM"]].head(20)

,LSOA21NM,WD22NM
14501,"Bournemouth, Christchurch and Poole 009B",Moordown
14505,"Bournemouth, Christchurch and Poole 046B",Hamworthy
14506,"Bournemouth, Christchurch and Poole 015C",Winton East
14507,"Bournemouth, Christchurch and Poole 009C",Moordown
14508,"Bournemouth, Christchurch and Poole 009D",Moordown
14509,"Bournemouth, Christchurch and Poole 027A",Queen's Park
14510,"Bournemouth, Christchurch and Poole 025D",Queen's Park
14511,"Bournemouth, Christchurch and Poole 038B",East Cliff & Springbourne
14512,"Bournemouth, Christchurch and Poole 025E",Queen's Park
14513,"Bournemouth, Christchurch and Poole 017B",Queen's Park


In [72]:
ward_df[ward_df["LSOA21NM"] == "Bournemouth, Christchurch and Poole 036A"][["LSOA21CD", "LSOA21NM", "WD22NM", "LTLA22NM"]]
# Check an example LSOA-to-ward match

,LSOA21CD,LSOA21NM,WD22NM,LTLA22NM
14663,E01015274,"Bournemouth, Christchurch and Poole 036A",Boscombe East & Pokesdown,"Bournemouth, Christchurch and Poole"


In [73]:
ward_lookup = ward_df[["LSOA21CD", "WD22NM"]].rename(columns={"LSOA21CD": "lsoa_code","WD22NM": "ward_name"}) 
assert ward_lookup["lsoa_code"].is_unique

In [74]:
# Add ward names using LSOA codes 
geo_final_csv = geo_final_csv.merge(
    ward_lookup,
    on="lsoa_code",
    how="left",
    validate="many_to_one",
    indicator=True)

print(geo_final_csv["_merge"].value_counts())

matched = geo_final_csv["location_status"] == "Matched English LSOA"

_merge
both          170569
left_only        837
right_only         0
Name: count, dtype: int64


In [75]:
# Validate ward join
matched = geo_final_csv["location_status"] == "Matched English LSOA"

assert len(geo_final_csv) == len(geo_final)
assert geo_final_csv["incident_count"].sum() == len(cleaned_df)
assert geo_final_csv.loc[matched, "ward_name"].notna().all()

# Remove the temporary join check column
geo_final_csv = geo_final_csv.drop(columns="_merge")

geo_final_csv[["lsoa_name", "ward_name", "incident_count"]].head(10)

,lsoa_name,ward_name,incident_count
0,"Bournemouth, Christchurch and Poole 035B",Boscombe East & Pokesdown,2
1,"Bournemouth, Christchurch and Poole 036A",Boscombe East & Pokesdown,3
2,"Bournemouth, Christchurch and Poole 035C",Boscombe East & Pokesdown,3
3,"Bournemouth, Christchurch and Poole 035D",Boscombe East & Pokesdown,2
4,"Bournemouth, Christchurch and Poole 035E",Boscombe East & Pokesdown,7
5,"Bournemouth, Christchurch and Poole 037A",Boscombe West,9
6,"Bournemouth, Christchurch and Poole 037B",Boscombe West,6
7,"Bournemouth, Christchurch and Poole 038A",Bournemouth Central,3
8,"Bournemouth, Christchurch and Poole 041A",East Cliff & Springbourne,3
9,"Bournemouth, Christchurch and Poole 041B",East Cliff & Springbourne,6


In [76]:
# Extract local authority and LSOA identifier
geo_final_csv[["local_authority", "lsoa_identifier"]] = (geo_final_csv["lsoa_name"].str.extract(r"^(.*) (\d{3}[A-Z])$")) 
geo_final_csv.head(10)

,police_force,year,month,crime_category,incident_type,location_status,lsoa_code,lsoa_name,incident_count,lsoa_population_2022,lsoa_income_deprivation_rate,lsoa_monthly_rate_per_1000,force_population,force_income_deprivation_rate,ward_name,local_authority,lsoa_identifier
0,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015273,"Bournemouth, Christchurch and Poole 035B",2,2118.0,0.315,0.944287,797350,0.181604,Boscombe East & Pokesdown,"Bournemouth, Christchurch and Poole",035B
1,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015274,"Bournemouth, Christchurch and Poole 036A",3,1844.0,0.179,1.626898,797350,0.181604,Boscombe East & Pokesdown,"Bournemouth, Christchurch and Poole",036A
2,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015275,"Bournemouth, Christchurch and Poole 035C",3,1704.0,0.189,1.760563,797350,0.181604,Boscombe East & Pokesdown,"Bournemouth, Christchurch and Poole",035C
3,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015276,"Bournemouth, Christchurch and Poole 035D",2,1769.0,0.229,1.130582,797350,0.181604,Boscombe East & Pokesdown,"Bournemouth, Christchurch and Poole",035D
4,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015277,"Bournemouth, Christchurch and Poole 035E",7,2028.0,0.312,3.451677,797350,0.181604,Boscombe East & Pokesdown,"Bournemouth, Christchurch and Poole",035E
5,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015279,"Bournemouth, Christchurch and Poole 037A",9,1930.0,0.411,4.663212,797350,0.181604,Boscombe West,"Bournemouth, Christchurch and Poole",037A
6,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015280,"Bournemouth, Christchurch and Poole 037B",6,2148.0,0.352,2.793296,797350,0.181604,Boscombe West,"Bournemouth, Christchurch and Poole",037B
7,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015286,"Bournemouth, Christchurch and Poole 038A",3,2035.0,0.191,1.474201,797350,0.181604,Bournemouth Central,"Bournemouth, Christchurch and Poole",038A
8,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015288,"Bournemouth, Christchurch and Poole 041A",3,1945.0,0.440,1.542416,797350,0.181604,East Cliff & Springbourne,"Bournemouth, Christchurch and Poole",041A
9,Dorset Police,2024,2024-01,Anti-social behaviour,ASB,Matched English LSOA,E01015289,"Bournemouth, Christchurch and Poole 041B",6,2051.0,0.275,2.925402,797350,0.181604,East Cliff & Springbourne,"Bournemouth, Christchurch and Poole",041B


### 7.6 Geographical Validation
Final checks confirm that reporting combinations are unique, required fields are complete and incident totals match the cleaned dataset.

Additional checks ensure that matched English LSOAs have the population, deprivation and geographical information needed for reporting.

In [77]:
# Define final reporting grain
grain = [
    "police_force",
    "month",
    "crime_category",
    "location_status",
    "lsoa_code"]

# Check for duplicate reporting combinations
assert not geo_final_csv.duplicated(subset=grain).any()

assert geo_final_csv["incident_count"].sum() == len(cleaned_df)

# Ensure essential reporting fields are complete
required_columns = [
    "police_force",
    "month",
    "crime_category",
    "incident_count",
    "location_status",
    "force_population"]

assert geo_final_csv[required_columns].notna().all().all()

# Check geographical enrichment is complete for matched English LSOAs 
matched = geo_final_csv["location_status"] == "Matched English LSOA"
assert geo_final_csv.loc[matched,["lsoa_code", "ward_name", "lsoa_population_2022", "lsoa_income_deprivation_rate"]].notna().all().all()
assert geo_final_csv.loc[matched, ["local_authority", "lsoa_identifier"]].notna().all().all()

print("Final geographical validation passed.")
print("Reporting rows:", len(geo_final_csv))
print("Total incidents:", geo_final_csv["incident_count"].sum())

Final geographical validation passed.
Reporting rows: 171406
Total incidents: 436692


## 8. Final Export and Summary

The validated dataset is exported as a single CSV file for use in Power BI. The exported file is then checked to confirm the incident totals, reporting coverage and number of geographical areas.


In [78]:
geo_final_csv.to_csv("crime_reporting_dataset.csv",index=False)
print("Reporting dataset exported to CSV successfully.")

Reporting dataset exported to CSV successfully.


In [79]:
#Final export summary
export_check = pd.read_csv("crime_reporting_dataset.csv")

print("FINAL DATASET SUMMARY")
print(f"Reporting rows: {len(export_check):,}")
print(f"Total incidents: {export_check['incident_count'].sum():,}")
print(f"Police forces: {export_check['police_force'].nunique()}")
print(f"Crime categories: {export_check['crime_category'].nunique()}")
print(f"Reporting months: {export_check['month'].nunique()}")
print(f"Period: {export_check['month'].min()} to {export_check['month'].max()}")
print(f"Unique LSOAs: {export_check['lsoa_code'].nunique()}")
print(f"Local authorities: {export_check['local_authority'].nunique()}")

# Confirm it matches the final DF
assert len(export_check) == len(geo_final_csv)
assert export_check["incident_count"].sum() == len(cleaned_df)

print("\nExport validation passed.")

FINAL DATASET SUMMARY
Reporting rows: 171,406
Total incidents: 436,692
Police forces: 4
Crime categories: 14
Reporting months: 24
Period: 2024-01 to 2025-12
Unique LSOAs: 2528
Local authorities: 199

Export validation passed.
